In [2]:
# Install required packages
!pip install stable-baselines3[extra] gymnasium[box2d] --quiet --break-system-packages 2>/dev/null || \
!pip install stable-baselines3[extra] gymnasium[box2d] --quiet

import gymnasium as gym
import stable_baselines3
print(f"Gymnasium version: {gym.__version__}")
print(f"Stable-Baselines3 version: {stable_baselines3.__version__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 102.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 100.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 148.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 19.6 MB/s eta 0:00:00


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Gymnasium version: 1.3.0
Stable-Baselines3 version: 2.9.0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
import gymnasium as gym

# continuous=True: main engine and lateral engine take continuous values (like the drone action space)
env = gym.make("LunarLander-v3", continuous=True)

obs, info = env.reset(seed=0)
print(f"Observation shape: {obs.shape}")
print(f"Observation space: {env.observation_space}")
print(f"Action space: {env.action_space}")
print(f"Sample observation: {obs}")

env.close()

Observation shape: (8,)
Observation space: Box([ -2.5        -2.5       -10.        -10.         -6.2831855 -10.
  -0.         -0.       ], [ 2.5        2.5       10.        10.         6.2831855 10.
  1.         1.       ], (8,), float32)
Action space: Box(-1.0, 1.0, (2,), float32)
Sample observation: [ 0.00570612  1.3990337   0.5779653  -0.5282997  -0.0066053  -0.13091765
  0.          0.        ]


<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type swigvarlink has no __module__ attribute


In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv

def make_env(seed):
    def _init():
        env = gym.make("LunarLander-v3", continuous=True)
        env = Monitor(env)
        env.reset(seed=seed)
        return env
    return _init

# Single training run, seed=1 (this will become ensemble member 1 later)
train_env = DummyVecEnv([make_env(seed=1)])

model = PPO(
    "MlpPolicy",
    train_env,
    verbose=1,
    seed=1,
)

model.learn(total_timesteps=500_000, progress_bar=True)
model.save("ppo_lunarlander_seed1")

print("Training complete, model saved.")

Output()

/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

Görüntülenen çıkış son 5000 satıra kısaltıldı.
|    loss                 | 56.9         |
|    n_updates            | 160          |
|    policy_gradient_loss | -0.00512     |
|    std                  | 0.978        |
|    value_loss           | 156          |
------------------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 237         |
|    ep_rew_mean          | -47.6       |
| time/                   |             |
|    fps                  | 482         |
|    iterations           | 18          |
|    time_elapsed         | 76          |
|    total_timesteps      | 36864       |
| train/                  |             |
|    approx_kl            | 0.004966896 |
|    clip_fraction        | 0.0408      |
|    clip_range           | 0.2         |
|    entropy_loss         | -2.79       |
|    explained_variance   | 0.585       |
|    learning_rate        | 0.0003      |
|    loss              

Training complete, model saved.


In [ ]:
import numpy as np

def evaluate_policy_custom(model, n_episodes=50, seed_start=0):
    successes = 0
    crashes = 0
    timeouts = 0
    rewards = []

    for i in range(n_episodes):
        eval_env = gym.make("LunarLander-v3", continuous=True)
        obs, info = eval_env.reset(seed=seed_start + i)
        episode_reward = 0
        done = False

        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = eval_env.step(action)
            episode_reward += reward
            done = terminated or truncated

        rewards.append(episode_reward)

        if terminated and episode_reward < 0:
            crashes += 1
        elif terminated and episode_reward >= 200:
            successes += 1
        elif truncated:
            timeouts += 1

        eval_env.close()

    rewards = np.array(rewards)
    print(f"Episodes: {n_episodes}")
    print(f"Mean reward: {rewards.mean():.1f} +/- {rewards.std():.1f}")
    print(f"Success rate (reward >= 200): {successes/n_episodes*100:.1f}%")
    print(f"Crash rate: {crashes/n_episodes*100:.1f}%")
    print(f"Timeout rate: {timeouts/n_episodes*100:.1f}%")

    return rewards

rewards = evaluate_policy_custom(model, n_episodes=50, seed_start=1000)

Episodes: 50
Mean reward: 180.6 +/- 88.9
Success rate (reward >= 200): 70.0%
Crash rate: 8.0%
Timeout rate: 2.0%


In [ ]:
# Train ensemble members 2 through 8 (member 1 already done)

for seed in range(2, 9):
    print(f"\n{'='*50}")
    print(f"Training ensemble member: seed={seed}")
    print(f"{'='*50}\n")

    train_env_i = DummyVecEnv([make_env(seed=seed)])

    model_i = PPO(
        "MlpPolicy",
        train_env_i,
        verbose=0,
        seed=seed,
    )

    model_i.learn(total_timesteps=500_000, progress_bar=True)
    model_i.save(f"ppo_lunarlander_seed{seed}")

    print(f"\nMember {seed} saved: ppo_lunarlander_seed{seed}.zip")

print("\nAll ensemble members trained.")

Output()


Training ensemble member: seed=2



Output()


Member 2 saved: ppo_lunarlander_seed2.zip

Training ensemble member: seed=3



Output()

...

Output()


Member 6 saved: ppo_lunarlander_seed6.zip

Training ensemble member: seed=7



Output()


Member 7 saved: ppo_lunarlander_seed7.zip

Training ensemble member: seed=8




Member 8 saved: ppo_lunarlander_seed8.zip

All ensemble members trained.


In [3]:
from stable_baselines3 import PPO
import numpy as np
import gymnasium as gym

# Load all 8 ensemble members
ensemble = []
for seed in range(1, 9):
    m = PPO.load(f"ppo_lunarlander_seed{seed}")
    ensemble.append(m)

print(f"Loaded {len(ensemble)} ensemble members.")

def collect_episode_disagreement(ensemble, env_seed, max_steps=1000):
    """
    Run one episode using ensemble member 1 as the acting policy.
    At each step, query all 8 members' actions on the same observation
    to compute action disagreement, without letting the other members act.
    """
    env = gym.make("LunarLander-v3", continuous=True)
    obs, info = env.reset(seed=env_seed)

    step_disagreements = []
    total_reward = 0
    crashed = False

    for t in range(max_steps):
        # Get action from each ensemble member on the SAME obs
        actions = np.array([m.predict(obs, deterministic=True)[0] for m in ensemble])
        # actions shape: (8, 2) -- 8 members, 2 action dims (main, lateral engine)

        # Disagreement: std across ensemble, per action dimension, then mean
        disagreement = actions.std(axis=0).mean()
        step_disagreements.append(disagreement)

        # Acting policy = member 1 (arbitrary choice, consistent across episodes)
        act_action = actions[0]
        obs, reward, terminated, truncated, info = env.step(act_action)
        total_reward += reward

        if terminated or truncated:
            crashed = terminated and total_reward < 0
            break

    env.close()
    return {
        "seed": env_seed,
        "disagreement": np.array(step_disagreements),
        "total_reward": total_reward,
        "crashed": crashed,
        "episode_len": t + 1,
    }

# Quick test on a single episode first
result = collect_episode_disagreement(ensemble, env_seed=1000)
print(f"Episode length: {result['episode_len']}")
print(f"Total reward: {result['total_reward']:.1f}")
print(f"Crashed: {result['crashed']}")
print(f"Disagreement shape: {result['disagreement'].shape}")
print(f"Mean disagreement: {result['disagreement'].mean():.4f}")

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type swigvarlink has no __module__ attribute


Loaded 8 ensemble members.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Episode length: 317
Total reward: 252.5
Crashed: False
Disagreement shape: (317,)
Mean disagreement: 0.3070


In [4]:
import pickle
import time

def collect_dataset(ensemble, seed_range, max_steps=1000):
    dataset = []
    start = time.time()

    for i, s in enumerate(seed_range):
        result = collect_episode_disagreement(ensemble, env_seed=s, max_steps=max_steps)
        dataset.append(result)

        if (i + 1) % 50 == 0:
            elapsed = time.time() - start
            print(f"{i+1}/{len(seed_range)} episodes, {elapsed:.1f}s elapsed")

    return dataset

# Exploration set: seeds 1000-1299
exploration_data = collect_dataset(ensemble, range(1000, 1300))

# Save immediately so a session drop doesn't lose the work
with open("exploration_data.pkl", "wb") as f:
    pickle.dump(exploration_data, f)

crash_count = sum(1 for d in exploration_data if d["crashed"])
print(f"\nExploration set: {len(exploration_data)} episodes, {crash_count} crashes ({crash_count/len(exploration_data)*100:.1f}%)")

50/300 episodes, 90.5s elapsed
100/300 episodes, 180.9s elapsed
150/300 episodes, 269.8s elapsed
200/300 episodes, 354.8s elapsed
250/300 episodes, 450.4s elapsed
300/300 episodes, 537.0s elapsed

Exploration set: 300 episodes, 24 crashes (8.0%)


In [5]:
# Confirmatory set: seeds 2000-2299 (completely separate from exploration)
confirmatory_data = collect_dataset(ensemble, range(2000, 2300))

with open("confirmatory_data.pkl", "wb") as f:
    pickle.dump(confirmatory_data, f)

crash_count = sum(1 for d in confirmatory_data if d["crashed"])
print(f"\nConfirmatory set: {len(confirmatory_data)} episodes, {crash_count} crashes ({crash_count/len(confirmatory_data)*100:.1f}%)")

50/300 episodes, 86.6s elapsed
100/300 episodes, 178.1s elapsed
150/300 episodes, 261.0s elapsed
200/300 episodes, 348.5s elapsed
250/300 episodes, 439.5s elapsed
300/300 episodes, 527.0s elapsed

Confirmatory set: 300 episodes, 28 crashes (9.3%)


In [6]:
from sklearn.metrics import roc_auc_score
import numpy as np

def per_dim_disagreement_episode(ensemble, env_seed, max_steps=1000):
    """Same as before, but keep disagreement separate per action dimension
    instead of averaging across dims immediately."""
    env = gym.make("LunarLander-v3", continuous=True)
    obs, info = env.reset(seed=env_seed)

    dim_disagreements = []  # list of (2,) arrays: [main_engine_std, lateral_std]
    total_reward = 0

    for t in range(max_steps):
        actions = np.array([m.predict(obs, deterministic=True)[0] for m in ensemble])
        dim_std = actions.std(axis=0)  # shape (2,) -- per-dimension std across 8 members
        dim_disagreements.append(dim_std)

        act_action = actions[0]
        obs, reward, terminated, truncated, info = env.step(act_action)
        total_reward += reward

        if terminated or truncated:
            crashed = terminated and total_reward < 0
            break

    env.close()
    dim_disagreements = np.array(dim_disagreements)  # shape (T, 2)
    return {
        "seed": env_seed,
        "dim_disagreement": dim_disagreements,
        "total_reward": total_reward,
        "crashed": crashed,
    }

# Re-collect exploration set with per-dimension detail
exploration_dimdata = [per_dim_disagreement_episode(ensemble, s) for s in range(1000, 1300)]

# Episode-level summary: mean disagreement per dimension, per episode
main_engine_means = np.array([d["dim_disagreement"][:, 0].mean() for d in exploration_dimdata])
lateral_means = np.array([d["dim_disagreement"][:, 1].mean() for d in exploration_dimdata])
labels = np.array([1 if d["crashed"] else 0 for d in exploration_dimdata])

auroc_main = roc_auc_score(labels, main_engine_means)
auroc_lateral = roc_auc_score(labels, lateral_means)
auroc_total = roc_auc_score(labels, (main_engine_means + lateral_means) / 2)

print(f"Main engine disagreement AUROC: {auroc_main:.3f}")
print(f"Lateral engine disagreement AUROC: {auroc_lateral:.3f}")
print(f"Total (averaged) disagreement AUROC: {auroc_total:.3f}")

Main engine disagreement AUROC: 0.862
Lateral engine disagreement AUROC: 0.599
Total (averaged) disagreement AUROC: 0.825


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [7]:
# Re-collect confirmatory set with per-dimension detail (same seeds as before, 2000-2299)
confirmatory_dimdata = [per_dim_disagreement_episode(ensemble, s) for s in range(2000, 2300)]

main_engine_means_conf = np.array([d["dim_disagreement"][:, 0].mean() for d in confirmatory_dimdata])
lateral_means_conf = np.array([d["dim_disagreement"][:, 1].mean() for d in confirmatory_dimdata])
labels_conf = np.array([1 if d["crashed"] else 0 for d in confirmatory_dimdata])

auroc_main_conf = roc_auc_score(labels_conf, main_engine_means_conf)
auroc_lateral_conf = roc_auc_score(labels_conf, lateral_means_conf)

print(f"[CONFIRMATORY] Main engine disagreement AUROC: {auroc_main_conf:.3f}")
print(f"[CONFIRMATORY] Lateral engine disagreement AUROC: {auroc_lateral_conf:.3f}")

# Save both dimdata sets for later operating-curve analysis
import pickle
with open("exploration_dimdata.pkl", "wb") as f:
    pickle.dump(exploration_dimdata, f)
with open("confirmatory_dimdata.pkl", "wb") as f:
    pickle.dump(confirmatory_dimdata, f)

[CONFIRMATORY] Main engine disagreement AUROC: 0.854
[CONFIRMATORY] Lateral engine disagreement AUROC: 0.695


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [8]:
def operating_curve(disagreement_means, labels, thresholds):
    """
    For each threshold tau, compute:
    - catch_rate: fraction of actual crashes correctly flagged (disagreement > tau)
    - false_halt_rate: fraction of non-crashes incorrectly flagged
    """
    catch_rates = []
    false_halt_rates = []

    crashed_mask = labels == 1
    safe_mask = labels == 0

    for tau in thresholds:
        flagged = disagreement_means > tau
        catch_rate = flagged[crashed_mask].mean() if crashed_mask.sum() > 0 else 0
        false_halt_rate = flagged[safe_mask].mean() if safe_mask.sum() > 0 else 0
        catch_rates.append(catch_rate)
        false_halt_rates.append(false_halt_rate)

    return np.array(catch_rates), np.array(false_halt_rates)

# Threshold range based on observed data range
tau_range = np.linspace(
    main_engine_means.min(), main_engine_means.max(), 50
)

# Exploration curve
catch_exp, fhr_exp = operating_curve(main_engine_means, labels, tau_range)

# Confirmatory curve (same thresholds, new data)
catch_conf, fhr_conf = operating_curve(main_engine_means_conf, labels_conf, tau_range)

# Print a compact table at a few reference false-halt-rate levels
for target_fhr in [0.05, 0.10, 0.15, 0.20]:
    idx_exp = np.argmin(np.abs(fhr_exp - target_fhr))
    idx_conf = np.argmin(np.abs(fhr_conf - target_fhr))
    print(f"False halt ~{target_fhr:.0%}: "
          f"exploration catch={catch_exp[idx_exp]:.0%} (tau={tau_range[idx_exp]:.3f}), "
          f"confirmatory catch={catch_conf[idx_conf]:.0%} (tau={tau_range[idx_conf]:.3f})")

with open("operating_curve_data.pkl", "wb") as f:
    pickle.dump({
        "tau_range": tau_range,
        "catch_exp": catch_exp, "fhr_exp": fhr_exp,
        "catch_conf": catch_conf, "fhr_conf": fhr_conf,
    }, f)

False halt ~5%: exploration catch=29% (tau=0.394), confirmatory catch=36% (tau=0.369)
False halt ~10%: exploration catch=50% (tau=0.362), confirmatory catch=50% (tau=0.350)
False halt ~15%: exploration catch=67% (tau=0.343), confirmatory catch=64% (tau=0.337)
False halt ~20%: exploration catch=79% (tau=0.324), confirmatory catch=71% (tau=0.324)


In [9]:
import numpy as np

exp_lengths = np.array([len(d["dim_disagreement"]) for d in exploration_dimdata])
exp_labels = np.array([1 if d["crashed"] else 0 for d in exploration_dimdata])

print("Crashed episodes:")
print(f"  n={exp_labels.sum()}, mean length={exp_lengths[exp_labels==1].mean():.0f}, "
      f"min={exp_lengths[exp_labels==1].min()}, median={np.median(exp_lengths[exp_labels==1]):.0f}")

print("Non-crashed episodes:")
print(f"  n={(exp_labels==0).sum()}, mean length={exp_lengths[exp_labels==0].mean():.0f}, "
      f"min={exp_lengths[exp_labels==0].min()}, median={np.median(exp_lengths[exp_labels==0]):.0f}")

# Coverage at candidate checkpoints: what fraction of crashes survive to step C?
for C in [30, 50, 75, 100, 150]:
    coverage = (exp_lengths[exp_labels==1] > C).mean()
    print(f"C={C:3d}: coverage={coverage:.2f} "
          f"({(exp_lengths[exp_labels==1] > C).sum()}/{exp_labels.sum()} crashes reachable)")

Crashed episodes:
  n=24, mean length=307, min=248, median=289
Non-crashed episodes:
  n=276, mean length=368, min=170, median=338
C= 30: coverage=1.00 (24/24 crashes reachable)
C= 50: coverage=1.00 (24/24 crashes reachable)
C= 75: coverage=1.00 (24/24 crashes reachable)
C=100: coverage=1.00 (24/24 crashes reachable)
C=150: coverage=1.00 (24/24 crashes reachable)


In [10]:
def causal_checkpoint_sweep(dimdata, labels, C, warmup=30, tau_range=None):
    """
    Causal rule: at step C, decide using only mean(main_engine_disagreement[warmup:C]).
    Strictly backward-looking -- no information from after step C.
    """
    scores = []
    for d in dimdata:
        main_dis = d["dim_disagreement"][:, 0]
        if len(main_dis) <= C:
            # Episode ended before checkpoint -- cannot decide (not applicable here)
            scores.append(np.nan)
        else:
            scores.append(main_dis[warmup:C].mean())
    scores = np.array(scores)

    valid = ~np.isnan(scores)
    if tau_range is None:
        tau_range = np.linspace(scores[valid].min(), scores[valid].max(), 60)

    results = []
    for tau in tau_range:
        fired = (scores > tau) & valid
        tp = (fired & (labels == 1)).sum()
        fp = (fired & (labels == 0)).sum()
        recall = tp / (labels == 1).sum()
        false_alarm = fp / (labels == 0).sum()
        precision = tp / fired.sum() if fired.sum() > 0 else 0
        results.append((tau, recall, false_alarm, precision, fired.sum()))

    return scores, np.array(results)

# AUROC of the causal score itself, at several checkpoints
from sklearn.metrics import roc_auc_score

for C in [50, 75, 100, 150, 200]:
    scores_exp, _ = causal_checkpoint_sweep(exploration_dimdata, exp_labels, C=C)
    auroc = roc_auc_score(exp_labels, scores_exp)
    print(f"C={C:3d}: causal AUROC (exploration) = {auroc:.3f}")

C= 50: causal AUROC (exploration) = 0.674
C= 75: causal AUROC (exploration) = 0.679
C=100: causal AUROC (exploration) = 0.692
C=150: causal AUROC (exploration) = 0.710


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


ValueError: Input contains NaN.

In [11]:
for C in [50, 75, 100, 150, 200, 250]:
    scores_exp, _ = causal_checkpoint_sweep(exploration_dimdata, exp_labels, C=C)
    valid = ~np.isnan(scores_exp)
    n_dropped = (~valid).sum()
    coverage = valid.mean()
    auroc = roc_auc_score(exp_labels[valid], scores_exp[valid])
    print(f"C={C:3d}: causal AUROC = {auroc:.3f}, coverage = {coverage:.2f} "
          f"({n_dropped} episodes ended before checkpoint)")

C= 50: causal AUROC = 0.674, coverage = 1.00 (0 episodes ended before checkpoint)
C= 75: causal AUROC = 0.679, coverage = 1.00 (0 episodes ended before checkpoint)
C=100: causal AUROC = 0.692, coverage = 1.00 (0 episodes ended before checkpoint)
C=150: causal AUROC = 0.710, coverage = 1.00 (0 episodes ended before checkpoint)
C=200: causal AUROC = 0.705, coverage = 0.98 (5 episodes ended before checkpoint)
C=250: causal AUROC = 0.713, coverage = 0.97 (9 episodes ended before checkpoint)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [12]:
C_FROZEN = 150
WARMUP = 30

conf_labels = np.array([1 if d["crashed"] else 0 for d in confirmatory_dimdata])

# Compute causal scores for both sets
scores_exp, _ = causal_checkpoint_sweep(exploration_dimdata, exp_labels, C=C_FROZEN, warmup=WARMUP)
scores_conf, _ = causal_checkpoint_sweep(confirmatory_dimdata, conf_labels, C=C_FROZEN, warmup=WARMUP)

valid_exp = ~np.isnan(scores_exp)
valid_conf = ~np.isnan(scores_conf)

auroc_exp = roc_auc_score(exp_labels[valid_exp], scores_exp[valid_exp])
auroc_conf = roc_auc_score(conf_labels[valid_conf], scores_conf[valid_conf])

print(f"Causal AUROC (C={C_FROZEN}):")
print(f"  Exploration : {auroc_exp:.3f}")
print(f"  Confirmatory: {auroc_conf:.3f}")
print()

# Shared threshold grid, derived from exploration only
tau_grid = np.linspace(scores_exp[valid_exp].min(), scores_exp[valid_exp].max(), 60)

def curve_at(scores, labels, valid, tau_grid):
    out = []
    for tau in tau_grid:
        fired = (scores > tau) & valid
        tp = (fired & (labels == 1)).sum()
        fp = (fired & (labels == 0)).sum()
        recall = tp / (labels == 1).sum()
        false_alarm = fp / (labels == 0).sum()
        precision = tp / fired.sum() if fired.sum() > 0 else 0.0
        out.append((tau, recall, false_alarm, precision))
    return np.array(out)

curve_exp = curve_at(scores_exp, exp_labels, valid_exp, tau_grid)
curve_conf = curve_at(scores_conf, conf_labels, valid_conf, tau_grid)

print("Causal operating points (threshold grid from exploration only):")
print(f"{'target FA':>10} | {'exp recall':>11} {'exp prec':>9} | {'conf recall':>12} {'conf prec':>10}")
for target_fa in [0.05, 0.10, 0.15, 0.20, 0.30]:
    i_e = np.argmin(np.abs(curve_exp[:, 2] - target_fa))
    i_c = np.argmin(np.abs(curve_conf[:, 2] - target_fa))
    print(f"{target_fa:>10.0%} | {curve_exp[i_e,1]:>11.0%} {curve_exp[i_e,3]:>9.0%} | "
          f"{curve_conf[i_c,1]:>12.0%} {curve_conf[i_c,3]:>10.0%}")

print(f"\nBase rate: exploration {exp_labels.mean():.1%}, confirmatory {conf_labels.mean():.1%}")

Causal AUROC (C=150):
  Exploration : 0.710
  Confirmatory: 0.591

Causal operating points (threshold grid from exploration only):
 target FA |  exp recall  exp prec |  conf recall  conf prec
        5% |          0%        0% |           4%         7%
       10% |         17%       12% |           7%         7%
       15% |         25%       14% |          18%        11%
       20% |         42%       16% |          21%        11%
       30% |         54%       14% |          50%        14%

Base rate: exploration 8.0%, confirmatory 9.3%


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [13]:
# For crashed episodes, align disagreement by "steps remaining until crash"
# instead of by absolute step index. This reveals WHEN the signal appears.

def time_to_failure_profile(dimdata, labels, lookback=250):
    """
    For each crashed episode, extract main-engine disagreement indexed by
    steps-remaining-before-termination. Non-crashed episodes are aligned
    the same way (by steps before their own termination) as a control.
    """
    crashed_profiles = []
    safe_profiles = []

    for d, lab in zip(dimdata, labels):
        main_dis = d["dim_disagreement"][:, 0]
        T = len(main_dis)
        if T < lookback:
            continue
        # Last `lookback` steps, so index -1 = final step before termination
        tail = main_dis[-lookback:]
        if lab == 1:
            crashed_profiles.append(tail)
        else:
            safe_profiles.append(tail)

    return np.array(crashed_profiles), np.array(safe_profiles)

crashed_prof, safe_prof = time_to_failure_profile(exploration_dimdata, exp_labels, lookback=250)

print(f"Crashed profiles: {crashed_prof.shape}, Safe profiles: {safe_prof.shape}\n")

print(f"{'steps before end':>17} | {'crashed':>9} | {'safe':>9} | {'gap':>7}")
for window_start, window_end, label in [
    (0, 25, "250-225"), (25, 50, "225-200"), (50, 100, "200-150"),
    (100, 150, "150-100"), (150, 200, "100-50"), (200, 230, "50-20"),
    (230, 250, "20-0"),
]:
    c_mean = crashed_prof[:, window_start:window_end].mean()
    s_mean = safe_prof[:, window_start:window_end].mean()
    print(f"{label:>17} | {c_mean:>9.4f} | {s_mean:>9.4f} | {c_mean - s_mean:>+7.4f}")

Crashed profiles: (23, 250), Safe profiles: (268, 250)

 steps before end |   crashed |      safe |     gap
          250-225 |    0.4534 |    0.2946 | +0.1589
          225-200 |    0.4336 |    0.2693 | +0.1644
          200-150 |    0.3657 |    0.2408 | +0.1250
          150-100 |    0.3171 |    0.2493 | +0.0678
           100-50 |    0.3050 |    0.2596 | +0.0454
            50-20 |    0.3193 |    0.0990 | +0.2203
             20-0 |    0.4472 |    0.0648 | +0.3824


In [14]:
def window_score(dimdata, start, end):
    """Causal score using ONLY steps [start:end]. Episodes shorter than `end` are NaN."""
    scores = []
    for d in dimdata:
        main_dis = d["dim_disagreement"][:, 0]
        if len(main_dis) < end:
            scores.append(np.nan)
        else:
            scores.append(main_dis[start:end].mean())
    return np.array(scores)

print(f"{'window':>12} | {'exp AUROC':>10} | {'conf AUROC':>11} | {'drift':>7} | {'coverage':>9}")
print("-" * 62)

windows = [(0, 50), (0, 80), (20, 60), (30, 80), (30, 100), (50, 100),
           (50, 150), (80, 150), (100, 200)]

for start, end in windows:
    s_exp = window_score(exploration_dimdata, start, end)
    s_conf = window_score(confirmatory_dimdata, start, end)
    v_exp, v_conf = ~np.isnan(s_exp), ~np.isnan(s_conf)

    a_exp = roc_auc_score(exp_labels[v_exp], s_exp[v_exp])
    a_conf = roc_auc_score(conf_labels[v_conf], s_conf[v_conf])
    cov = v_conf.mean()

    print(f"{f'[{start}:{end}]':>12} | {a_exp:>10.3f} | {a_conf:>11.3f} | "
          f"{a_conf - a_exp:>+7.3f} | {cov:>9.2f}")

      window |  exp AUROC |  conf AUROC |   drift |  coverage
--------------------------------------------------------------
      [0:50] |      0.655 |       0.460 |  -0.195 |      1.00
      [0:80] |      0.705 |       0.481 |  -0.224 |      1.00
     [20:60] |      0.704 |       0.480 |  -0.224 |      1.00
     [30:80] |      0.679 |       0.494 |  -0.186 |      1.00
    [30:100] |      0.692 |       0.543 |  -0.149 |      1.00
    [50:100] |      0.697 |       0.592 |  -0.105 |      1.00
    [50:150] |      0.702 |       0.619 |  -0.082 |      1.00
    [80:150] |      0.689 |       0.637 |  -0.052 |      1.00
   [100:200] |      0.664 |       0.662 |  -0.002 |      0.99


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
